# 💻 Chapter 15: A/B Testing — Design & Analysis
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** Your A/B test says the new button color is 15% better with p=0.03.
> Should you ship it? The answer depends on things most teams never check.

**🎯 Objectives:** Design statistically valid A/B tests; avoid common pitfalls; analyze results correctly.

## 📖 Section 1 — Concept Review

### The A/B Testing Framework
1. **Metric**: What are you measuring? (CTR, conversion rate, revenue)
2. **Null hypothesis H₀**: No difference between A and B
3. **Sample size**: Calculate BEFORE running the test
4. **Duration**: Run to completion, don't peek
5. **Analysis**: Two-proportion z-test or t-test

### Sample Size Calculation
For comparing two proportions:
$$n = \frac{(z_{\alpha/2} + z_\beta)^2 (p_1(1-p_1) + p_2(1-p_2))}{(p_1 - p_2)^2}$$

### The Most Common Mistakes
1. **Peeking**: Stopping early when you see p<0.05 — inflates Type I error
2. **Multiple comparisons**: Testing 20 metrics without correction
3. **Too small n**: Underpowered test misses real effects
4. **Not specifying MDE**: Minimum Detectable Effect must be set in advance
5. **Novelty effect**: Users respond differently to anything new

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import pandas as pd
rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid")

# ── Sample Size Calculator ──
def ab_sample_size(p_baseline, mde, alpha=0.05, power=0.80):
    # Required sample size per variant.
    p1 = p_baseline
    p2 = p_baseline + mde
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(power)
    n = ((z_alpha + z_beta)**2 * (p1*(1-p1) + p2*(1-p2))) / (p1-p2)**2
    return int(np.ceil(n))

print("📊 A/B Test Sample Size Calculator")
print(f"{'Baseline CTR':>14} {'MDE':>8} {'Required n/variant':>20} {'Total n':>10}")
print("-" * 56)
for p_base in [0.02, 0.05, 0.10, 0.20]:
    for mde in [0.005, 0.01, 0.02]:
        if mde/p_base > 0.5:
            n = ab_sample_size(p_base, mde)
            total = 2 * n
            print(f"{p_base:>14.1%} {mde:>8.1%} {n:>20,} {total:>10,}")
print()
print("💡 Key insight: Large baselines AND small MDEs need HUGE samples!")

In [ ]:
# ── The Peeking Problem ──
rng = np.random.default_rng(42)

def run_ab_test_with_peeking(true_p_a, true_p_b, n_max, alpha=0.05):
    # Simulate peeking: check p-value after every 10 observations.
    data_a, data_b = [], []
    peeked_significant = False

    for _ in range(n_max):
        data_a.append(rng.random() < true_p_a)
        data_b.append(rng.random() < true_p_b)

        if len(data_a) >= 20 and len(data_a) % 10 == 0:
            conv_a = np.mean(data_a); conv_b = np.mean(data_b)
            n_a = len(data_a); n_b = len(data_b)
            # Two-proportion z-test
            p_pool = (sum(data_a) + sum(data_b)) / (n_a + n_b)
            se = np.sqrt(p_pool*(1-p_pool)*(1/n_a + 1/n_b))
            if se > 0:
                z = (conv_a - conv_b) / se
                p_val = 2 * (1 - stats.norm.cdf(abs(z)))
                if p_val < alpha:
                    peeked_significant = True
                    break

    # Final test without peeking
    _, p_final = stats.ttest_ind(data_a, data_b)
    return peeked_significant, p_final < alpha

# Simulate: no TRUE difference (H₀ true)
n_sims = 1000
peek_fp = 0; nopeak_fp = 0
for _ in range(n_sims):
    peek, no_peek = run_ab_test_with_peeking(0.05, 0.05, 1000)
    if peek: peek_fp += 1
    if no_peek: nopeak_fp += 1

print(f"🚨 Peeking Problem Demo (H₀ is TRUE — no real difference)")
print(f"  False positive rate WITH peeking:    {peek_fp/n_sims:.1%}")
print(f"  False positive rate WITHOUT peeking: {nopeak_fp/n_sims:.1%}")
print(f"  Target α = 5.0%")
print()
print("💥 Peeking inflates FP rate from 5% to much higher!")
print("   Always run tests to planned completion.")

In [ ]:
# ── Complete A/B Test Analysis ──
rng = np.random.default_rng(42)

# Simulate experiment results
n_per_variant = 5000
true_rate_A = 0.050  # control (current)
true_rate_B = 0.058  # treatment (new button)
mde = 0.008  # minimum detectable effect we set in advance

conversions_A = rng.binomial(1, true_rate_A, n_per_variant)
conversions_B = rng.binomial(1, true_rate_B, n_per_variant)

conv_A = conversions_A.mean()
conv_B = conversions_B.mean()
lift = (conv_B - conv_A) / conv_A

# Two-proportion z-test
p_pool = (conversions_A.sum() + conversions_B.sum()) / (n_per_variant * 2)
se = np.sqrt(p_pool * (1-p_pool) * 2/n_per_variant)
z = (conv_B - conv_A) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

alpha = 0.05
print("📊 A/B Test Results Report")
print("=" * 50)
print(f"  Variant A (control):  {conv_A:.4f} ({conversions_A.sum()} / {n_per_variant})")
print(f"  Variant B (treatment):{conv_B:.4f} ({conversions_B.sum()} / {n_per_variant})")
print(f"  Observed lift:        {lift:+.2%}")
print(f"  Z-statistic:          {z:.4f}")
print(f"  p-value:              {p_value:.4f}")
print()
if p_value < alpha and abs(conv_B - conv_A) >= mde:
    print(f"  ✅ SHIP IT: Statistically significant AND practically meaningful")
elif p_value < alpha:
    print(f"  ⚠️  Significant but effect size < MDE — may not be worth shipping")
else:
    print(f"  ❌ Not significant — don't ship")

# Confidence interval for the difference
ci_low = (conv_B - conv_A) - 1.96*se
ci_high = (conv_B - conv_A) + 1.96*se
print(f"
  95% CI for difference: ({ci_low:.4f}, {ci_high:.4f})")
print(f"  MDE was: {mde:.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
variants = ['Control (A)', 'Treatment (B)']
rates = [conv_A, conv_B]
errors = [1.96 * np.sqrt(r*(1-r)/n_per_variant) for r in rates]
colors = ['#3498db', '#27ae60' if p_value < alpha else '#e74c3c']

axes[0].bar(variants, rates, color=colors, alpha=0.7, yerr=errors, capsize=5)
axes[0].set_ylabel('Conversion Rate')
axes[0].set_title(f'A/B Test: p={p_value:.4f}', fontweight='bold')
for i, (r, e) in enumerate(zip(rates, errors)):
    axes[0].text(i, r+e+0.001, f'{r:.3%}', ha='center', fontweight='bold')

# Null distribution vs observed
x = np.linspace(-5, 5, 300)
axes[1].plot(x, stats.norm.pdf(x), lw=3, color='#3498db', label='Null distribution')
axes[1].fill_between(x, stats.norm.pdf(x), where=(abs(x) >= abs(z)), color='#e74c3c', alpha=0.4, label=f'p={p_value:.4f}')
axes[1].axvline(z, color='#e74c3c', lw=2, linestyle='--', label=f'z={z:.2f}')
axes[1].set_title('Hypothesis Test: Z-distribution', fontweight='bold')
axes[1].set_xlabel('Z'); axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('ch15_ab_test.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Coding Challenges

**Challenge 1:** Write a function `required_sample_size(p1, p2, alpha, power)`.
Test it for: p1=0.10, p2=0.12, α=0.05, power=0.80.

**Challenge 2:** Your A/B test has 5 metrics. If each uses α=0.05, what's the family-wise error rate?
Apply Bonferroni correction and recalculate.

**Challenge 3:** Implement a simple Bayesian A/B test using Beta-Binomial conjugacy.
Report P(B > A) and credible intervals.

<details><summary>Solutions</summary>

**C1:** n ≈ 3,560 per variant.

**C2:** FWER = 1-(0.95)^5 ≈ 0.226. Bonferroni: use α/5 = 0.01 per test.

**C3:** See Bayesian code below.
</details>

In [ ]:
# Bayesian A/B Test
from scipy.stats import beta

# Prior: Beta(1,1) = uniform
alpha_prior = 1; beta_prior = 1

# Observed
conv_a_obs = 250; visits_a = 5000
conv_b_obs = 290; visits_b = 5000

# Posterior: Beta(alpha + conversions, beta + non-conversions)
post_a = beta(alpha_prior + conv_a_obs, beta_prior + visits_a - conv_a_obs)
post_b = beta(alpha_prior + conv_b_obs, beta_prior + visits_b - conv_b_obs)

# P(B > A) via sampling
rng2 = np.random.default_rng(42)
samples_a = post_a.rvs(100_000, random_state=42)
samples_b = post_b.rvs(100_000, random_state=43)
p_b_better = (samples_b > samples_a).mean()

print(f"Bayesian A/B Test:")
print(f"  P(B > A) = {p_b_better:.3f}")
print(f"  P(B > A by > 10%) = {((samples_b / samples_a - 1) > 0.10).mean():.3f}")

x = np.linspace(0.03, 0.08, 300)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, post_a.pdf(x), lw=3, color='#3498db', label=f'Control (A): {conv_a_obs}/{visits_a}')
ax.plot(x, post_b.pdf(x), lw=3, color='#27ae60', label=f'Treatment (B): {conv_b_obs}/{visits_b}')
ax.fill_between(x, post_a.pdf(x), alpha=0.2, color='#3498db')
ax.fill_between(x, post_b.pdf(x), alpha=0.2, color='#27ae60')
ax.set_title(f'Bayesian A/B Test: P(B>A) = {p_b_better:.1%}', fontweight='bold')
ax.set_xlabel('Conversion Rate'); ax.set_ylabel('Posterior Density')
ax.legend(); plt.tight_layout()
plt.savefig('ch15_bayesian_ab.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎯 Recap
1. Calculate sample size BEFORE the experiment using MDE, α, and power.
2. Never peek — sequential testing frameworks exist for this.
3. Statistical significance ≠ practical significance — check effect size vs MDE.

**Next:** [Chapter 16 — Bayesian Updating in Practice]